# 🧠 DSA Lab 10: Stacks: LIFO, Implementations and Applications
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 10 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: The Stack ADT

An **Abstract Data Type (ADT)** describes **what operations** a structure supports, not **how** it is implemented. The stack ADT has these operations:

| Operation | Meaning | Time |
|---|---|---|
| `push(x)` | Add x on the top | O(1) |
| `pop()` | Remove and return the top element | O(1) |
| `peek()` / `top()` | Return the top element without removing it | O(1) |
| `is_empty()` | Is the stack empty? | O(1) |
| `size()` | Number of elements | O(1) |

| Error | When |
|---|---|
| Stack **underflow** | `pop()` or `peek()` on an empty stack |
| Stack **overflow** | `push()` on a full fixed-size stack |

**Stack operations step by step**

```text
push(10)      push(20)      push(30)      pop() -> 30   peek() -> 20
                            | 30 | <-top
              | 20 | <-top  | 20 |        | 20 | <-top  | 20 | <-top
| 10 | <-top  | 10 |        | 10 |        | 10 |        | 10 |
+----+        +----+        +----+        +----+        +----+
```

## 🧱 Part B: Implementations

### 1. Using a Python List (Dynamic Array)

The **end** of a Python list is the top of the stack: `append` and `pop()` at the end are O(1) amortised. Never use the front (`insert(0, x)`, `pop(0)`), which is O(n).

In [ ]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, x):
        self._items.append(x)

    def pop(self):
        if self.is_empty():
            raise IndexError("stack underflow")
        return self._items.pop()

    def peek(self):
        if self.is_empty():
            raise IndexError("stack is empty")
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0

    def size(self):
        return len(self._items)

    def __str__(self):
        return "bottom " + str(self._items) + " top"

s = Stack()
for x in [10, 20, 30]:
    s.push(x)
print(s, "| pop:", s.pop(), "| peek:", s.peek(), "| size:", s.size())

### 2. Fixed-Size Array Stack (Overflow and Underflow)

In C and embedded systems, the stack has a fixed capacity. We keep an index `top` (−1 when empty).

In [ ]:
class ArrayStack:
    def __init__(self, capacity):
        self.data = [None] * capacity
        self.top = -1

    def push(self, x):
        if self.top == len(self.data) - 1:
            raise OverflowError("stack overflow")
        self.top += 1
        self.data[self.top] = x

    def pop(self):
        if self.top == -1:
            raise IndexError("stack underflow")
        x = self.data[self.top]
        self.data[self.top] = None
        self.top -= 1
        return x

a = ArrayStack(2)
a.push("A"); a.push("B")
try:
    a.push("C")
except OverflowError as e:
    print("Error:", e)
print(a.pop(), a.pop())

### 3. Linked-List Stack

The **head** of a singly linked list is the top: `push_front` and `pop_front` are O(1) and the stack never needs resizing.

In [ ]:
class LinkedStack:
    class _Node:
        def __init__(self, data, next):
            self.data, self.next = data, next

    def __init__(self):
        self.head = None
        self.count = 0

    def push(self, x):
        self.head = self._Node(x, self.head)
        self.count += 1

    def pop(self):
        if self.head is None:
            raise IndexError("stack underflow")
        x = self.head.data
        self.head = self.head.next
        self.count -= 1
        return x

    def peek(self):
        return None if self.head is None else self.head.data

ls = LinkedStack()
for ch in "DSA":
    ls.push(ch)
print(ls.pop() + ls.pop() + ls.pop())     # reversed: ASD

| Implementation | push/pop | Pros | Cons |
|---|---|---|---|
| Python list | O(1) amortised | Simplest, fast, cache-friendly | Occasional resize |
| Fixed array | O(1) | No resizing, predictable memory | Overflow when full |
| Linked list | O(1) | Never full, no resize | Extra memory per node |

## 🧩 Part C: Classic Applications

### Balanced Brackets

Scan the expression. Push every **opening** bracket. For a **closing** bracket, the top of the stack must be the matching opening bracket; pop it. At the end, the stack must be empty.

In [ ]:
def is_balanced(expr):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in expr:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack

for e in ["{[a + b] * (c - d)}", "(a + b]", "((x)", "print(arr[i])"]:
    print(f"{e:<22} -> {is_balanced(e)}")

### Infix, Prefix and Postfix

| Notation | Example | Used by |
|---|---|---|
| Infix | A + B * C | Humans; needs precedence rules and brackets |
| Prefix (Polish) | + A * B C | Some compilers, LISP |
| Postfix (Reverse Polish) | A B C * + | Calculators, stack machines; no brackets needed |

### Evaluating a Postfix Expression

Read tokens left to right. **Number** → push. **Operator** → pop two numbers (the first pop is the **right** operand), apply the operator, push the result. The final value on the stack is the answer.

| Token | Action | Stack |
|---|---|---|
| 5 | push | [5] |
| 3 | push | [5, 3] |
| 2 | push | [5, 3, 2] |
| * | 3 × 2 = 6 | [5, 6] |
| + | 5 + 6 = 11 | [11] |
| 4 | push | [11, 4] |
| - | 11 − 4 = 7 | [7] |

In [ ]:
def eval_postfix(tokens):
    stack = []
    for t in tokens.split():
        if t in "+-*/":
            right = stack.pop()
            left = stack.pop()
            if t == "+": stack.append(left + right)
            elif t == "-": stack.append(left - right)
            elif t == "*": stack.append(left * right)
            else: stack.append(left / right)
        else:
            stack.append(float(t))
    return stack.pop()

print(eval_postfix("5 3 2 * + 4 -"))      # 5 + 3*2 - 4 = 7

### Infix to Postfix (Shunting-Yard Algorithm)

Rules: operands go straight to the output. `(` is pushed. `)` pops operators to the output until `(`. For an operator, first pop operators of **higher or equal precedence** (for left-associative operators), then push it. At the end, pop everything.

In [ ]:
PREC = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}

def infix_to_postfix(expr):
    out, stack = [], []
    for t in expr.split():
        if t == "(":
            stack.append(t)
        elif t == ")":
            while stack[-1] != "(":
                out.append(stack.pop())
            stack.pop()                          # discard "("
        elif t in PREC:
            while (stack and stack[-1] != "(" and
                   (PREC[stack[-1]] > PREC[t] or (PREC[stack[-1]] == PREC[t] and t != "^"))):
                out.append(stack.pop())
            stack.append(t)
        else:
            out.append(t)                        # operand
    while stack:
        out.append(stack.pop())
    return " ".join(out)

print(infix_to_postfix("A + B * C"))
print(infix_to_postfix("( A + B ) * C - D / E"))
print(infix_to_postfix("2 ^ 3 ^ 2"))             # ^ is right-associative

## 📉 Part D: Monotonic Stack

A **monotonic stack** keeps its elements in increasing or decreasing order. It solves “**next greater / smaller element**” problems in **O(n)** instead of O(n²), because each element is pushed and popped at most once.

**Problem:** for each day’s share price, find the next day with a **higher** price (−1 if none).

In [ ]:
def next_greater(prices):
    result = [-1] * len(prices)
    stack = []                       # indexes of days still waiting for a higher price
    for i, p in enumerate(prices):
        while stack and prices[stack[-1]] < p:
            result[stack.pop()] = p  # today is the answer for that earlier day
        stack.append(i)
    return result

prices = [100, 80, 120, 90, 95, 130]
print(next_greater(prices))

## 📞 Part E: The Call Stack and Recursion

Python itself uses a stack to run functions. Any recursive function can be rewritten using your own stack, which avoids Python’s recursion limit.

In [ ]:
def factorial_with_stack(n):
    stack = []
    while n > 0:                 # "calls": push pending work
        stack.append(n)
        n -= 1
    result = 1
    while stack:                 # "returns": pop and combine
        result *= stack.pop()
    return result

import traceback
def a(): b()
def b(): print("".join(traceback.format_stack(limit=3)[:-1]))
print(factorial_with_stack(10))
a()                              # shows the call stack: a -> b

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <stack>
#include <string>
using namespace std;

bool isBalanced(const string& s) {
    stack<char> st;
    for (char c : s) {
        if (c == '(' || c == '[' || c == '{') st.push(c);
        else if (c == ')' || c == ']' || c == '}') {
            if (st.empty()) return false;
            char t = st.top(); st.pop();
            if ((c == ')' && t != '(') || (c == ']' && t != '[') || (c == '}' && t != '{'))
                return false;
        }
    }
    return st.empty();
}

int main() {
    stack<int> s;              // std::stack: push, pop, top, empty, size
    s.push(10); s.push(20);
    cout << s.top() << endl;   // 20
    s.pop();                   // note: pop() returns void in C++
    cout << isBalanced("{[()]}") << isBalanced("(]") << endl;   // 10
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Reverse Words with a Stack
Write `reverse_words(sentence)` that reverses the **order of words** using a stack (push every word, then pop). Do not use slicing or `reversed()`.

```text
"Data Structures are fun" → "fun are Structures Data"
```

In [ ]:
def reverse_words(sentence):
    stack = []
    # TODO: push words, then pop them into a result list
    pass

In [ ]:
# Test cell: run after your solution
assert reverse_words("Data Structures are fun") == "fun are Structures Data"
assert reverse_words("one") == "one"
assert reverse_words("") == ""
print("✅ Task 1 passed")

### Task 2 🟢 HTML Tag Checker
Web browsers must check that HTML tags are properly nested. Given a list of tags like `"<p>"` and `"</p>"`, return `True` if every closing tag matches the most recent unclosed opening tag.

Hint: same idea as balanced brackets; a closing tag starts with `</`.

```text
["<html>", "<body>", "<p>", "</p>", "</body>", "</html>"] → True
["<b>", "<i>", "</b>", "</i>"] → False
```

In [ ]:
def tags_valid(tags):
    stack = []
    # TODO: push opening tag names; check closing tags against the top
    pass

In [ ]:
# Test cell: run after your solution
assert tags_valid(["<html>", "<body>", "<p>", "</p>", "</body>", "</html>"]) is True
assert tags_valid(["<b>", "<i>", "</b>", "</i>"]) is False
assert tags_valid(["<div>"]) is False
assert tags_valid(["</div>"]) is False
assert tags_valid([]) is True
print("✅ Task 2 passed")

### Task 3 🟡 Min Stack
Design `MinStack` with `push`, `pop`, `top` and `get_min`, **all O(1)**. Hint: keep a second stack that stores the current minimum at each level.

```text
push(5), push(2), push(7) → get_min 2; pop() → get_min 2; pop() → get_min 5
```

In [ ]:
class MinStack:
    def __init__(self):
        self.items = []
        self.mins = []

    def push(self, x):
        # TODO
        pass

    def pop(self):
        # TODO
        pass

    def top(self):
        # TODO
        pass

    def get_min(self):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
m = MinStack()
for x in [5, 2, 7]:
    m.push(x)
assert m.get_min() == 2 and m.top() == 7
m.pop(); assert m.get_min() == 2
m.pop(); assert m.get_min() == 5
m.push(1); m.push(1); m.pop(); assert m.get_min() == 1
print("✅ Task 3 passed")

### Task 4 🟡 Evaluate Infix Expressions
Combine `infix_to_postfix` and a postfix evaluator to write `calculate(expr)` for space-separated infix expressions with numbers, `+ - * /`, `^` and brackets. Return a float.

```text
calculate("( 2 + 3 ) * 4 - 6 / 2") → 17.0
```

In [ ]:
def calculate(expr):
    postfix = infix_to_postfix(expr)
    stack = []
    # TODO: evaluate postfix (support + - * / ^)
    pass

In [ ]:
# Test cell: run after your solution
assert calculate("( 2 + 3 ) * 4 - 6 / 2") == 17.0
assert calculate("2 ^ 3 ^ 2") == 512.0
assert calculate("10 - 4 - 3") == 3.0
assert calculate("7") == 7.0
print("✅ Task 4 passed")

### Task 5 🟡 Daily Temperatures
For each day in a list of Lahore temperatures, return how many days you must wait for a **warmer** day (0 if none). Use a monotonic stack for O(n).

```text
[31, 33, 29, 30, 35, 28] → [1, 3, 1, 1, 0, 0]
```

In [ ]:
def daily_temperatures(temps):
    wait = [0] * len(temps)
    stack = []
    # TODO: monotonic stack of indexes
    return wait

In [ ]:
# Test cell: run after your solution
assert daily_temperatures([31, 33, 29, 30, 35, 28]) == [1, 3, 1, 1, 0, 0]
assert daily_temperatures([30, 40, 50, 60]) == [1, 1, 1, 0]
assert daily_temperatures([]) == []
print("✅ Task 5 passed")

### Task 6 🔴 Text Editor with Undo and Redo
Build `Editor` with `type(text)` (append text), `delete(k)` (remove last k characters), `undo()` and `redo()`, and a `text` attribute.

Use **two stacks**: every change is pushed onto the undo stack; undo moves it to the redo stack; any new change clears the redo stack.

```text
type("Hello"); type(" World"); delete(6) → "Hello"
undo() → "Hello World"; undo() → "Hello"; redo() → "Hello World"
```

In [ ]:
class Editor:
    def __init__(self):
        self.text = ""
        self.undo_stack = []
        self.redo_stack = []

    def type(self, s):
        # TODO
        pass

    def delete(self, k):
        # TODO
        pass

    def undo(self):
        # TODO
        return self.text

    def redo(self):
        # TODO
        return self.text

In [ ]:
# Test cell: run after your solution
e = Editor()
e.type("Hello"); e.type(" World"); e.delete(6)
assert e.text == "Hello"
assert e.undo() == "Hello World" and e.undo() == "Hello" and e.redo() == "Hello World"
e.type("!"); assert e.redo() == "Hello World!"
assert e.undo() == "Hello World" and e.undo() == "Hello" and e.undo() == "" and e.undo() == ""
print("✅ Task 6 passed")

### Task 7 🔴 Largest Rectangle in a Histogram
A bar chart of daily rainfall has bar heights `h`. Find the area of the **largest rectangle** that fits entirely under the bars. Brute force is O(n²); use a monotonic increasing stack for **O(n)**.

Hint: when a bar is popped, it is the shortest bar of a rectangle that extends from the new stack top + 1 to the current index − 1.

```text
[2, 1, 5, 6, 2, 3] → 10      # bars 5 and 6, width 2
```

In [ ]:
def largest_rectangle(h):
    stack, best = [], 0
    # TODO: monotonic increasing stack; add a sentinel bar of height 0 at the end
    return best

In [ ]:
# Test cell: run after your solution
import random
def brute(h):
    return max([min(h[i:j + 1]) * (j - i + 1) for i in range(len(h)) for j in range(i, len(h))] or [0])
assert largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
assert largest_rectangle([]) == 0
assert largest_rectangle([4]) == 4
for _ in range(30):
    r = [random.randint(0, 9) for _ in range(random.randint(1, 15))]
    assert largest_rectangle(r) == brute(r)
print("✅ Task 7 passed")

## 🔬 Bonus: Solve a Maze with a Stack (Depth-First Search)

In [ ]:
# Bonus: run after completing the tasks
maze = ["S.#.....",
        ".##.###.",
        "....#...",
        "#.#.#.#.",
        "..#...#E"]
rows, cols = len(maze), len(maze[0])
start = (0, 0)
stack, came_from = [start], {start: None}
while stack:
    r, c = stack.pop()
    if maze[r][c] == "E":
        break
    for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] != "#" and (nr, nc) not in came_from:
            came_from[(nr, nc)] = (r, c)
            stack.append((nr, nc))
path, node = set(), (r, c)
while node:
    path.add(node); node = came_from[node]
for i in range(rows):
    print("".join("*" if (i, j) in path and maze[i][j] == "." else maze[i][j] for j in range(cols)))

---
## 🎉 Lab 10 Complete!

Next: **Lab 11: Queues, Circular Queues and Deques**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab10/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)